In [5]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [6]:
import sys
!apt-get install -y poppler-utils
!pip uninstall -y Pillow
!pip install pdf2image easyocr pandas
!pip install --upgrade --force-reinstall Pillow==9.3.0
!pip install pdfplumber
!pip install chromadb
!pip install -qU langchain-huggingface sentence-transformers langchain-google-genai langchain-community chromadb langgraph tavily-python pypdf langchain-text-splitters
!pip install reportlab arabic_reshaper python-bidi
!wget https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf -O Amiri-Regular.ttf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (24.02.0-1ubuntu9.9).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)


  Using cached Pillow-9.3.0.tar.gz (50.4 MB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
--2026-09-10 22:06:59--  https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Regular.ttf [following]
--2026-09-10 22:06:59--  https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Regular.ttf

In [7]:
[
  {
    "law_name": "قانون رقم 67 لسنة 2016 - القيمة المضافة",
    "article_number": "مادة (12)",
    "content": "لا يحق للمسجل خصم الضريبة إلا إذا كانت مدونة في فاتورة ضريبية إلكترونية معتمدة...",
    "category": "VAT_Deduction_Rules"
  },
  {
    "law_name": "قانون رقم 206 لسنة 2020 - الإجراءات الموحدة",
    "article_number": "مادة (37)",
    "content": "يجب على كل ممول أو مكلف إصدار فاتورة ضريبية أو إيصال إلكتروني يتضمن الرقم الضريبي...",
    "category": "E_Invoicing_Requirements"
  }
]

[{'law_name': 'قانون رقم 67 لسنة 2016 - القيمة المضافة',
  'article_number': 'مادة (12)',
  'content': 'لا يحق للمسجل خصم الضريبة إلا إذا كانت مدونة في فاتورة ضريبية إلكترونية معتمدة...',
  'category': 'VAT_Deduction_Rules'},
 {'law_name': 'قانون رقم 206 لسنة 2020 - الإجراءات الموحدة',
  'article_number': 'مادة (37)',
  'content': 'يجب على كل ممول أو مكلف إصدار فاتورة ضريبية أو إيصال إلكتروني يتضمن الرقم الضريبي...',
  'category': 'E_Invoicing_Requirements'}]

In [3]:
import glob
import json
import os
from pdf2image import convert_from_path
import easyocr

reader = easyocr.Reader(['ar'], gpu=True)


def process_scanned_pdf(pdf_path, law_title):
    print(f" جاري تحويل وتحليل: {pdf_path}")

    images = convert_from_path(pdf_path, dpi=150)
    data_chunks = []

    for i, img in enumerate(images):
        temp_img_path = f"temp_{i}.png"
        img.save(temp_img_path, "PNG")

        results = reader.readtext(temp_img_path, detail=0)
        page_text = " ".join(results)

        if page_text.strip():
            data_chunks.append({
                "law_name": law_title,
                "page_number": i + 1,
                "content": page_text.strip()
            })

        if os.path.exists(temp_img_path):
            os.remove(temp_img_path)

    return data_chunks


pdf_files = glob.glob("*.pdf")
all_chunks = []

print(f" إجمالي الملفات المكتشفة: {len(pdf_files)} ملفات\n")

for pdf_path in pdf_files:
    print("--------------------------------------------------")
    print(f" البدء في معالجة: {pdf_path}")

    law_title = pdf_path.replace(".pdf", "").replace("_", " ")
    file_chunks = process_scanned_pdf(pdf_path, law_title)
    all_chunks.extend(file_chunks)


with open("egyptian_tax_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=4)

print("\n تم الانتهاء من جميع الملفات بنجاح!")
print(f" إجمالي الفقرات المستخرجة: {len(all_chunks)} فقرة جاهزة للـ Vector Store.")

 إجمالي الملفات المكتشفة: 6 ملفات

--------------------------------------------------
 البدء في معالجة: vat-law_no.66-2017.pdf
 جاري تحويل وتحليل: vat-law_no.66-2017.pdf
--------------------------------------------------
 البدء في معالجة: vat-law_no.67-2016.pdf
 جاري تحويل وتحليل: vat-law_no.67-2016.pdf
--------------------------------------------------
 البدء في معالجة: law.no_.149.of_.2026.pdf
 جاري تحويل وتحليل: law.no_.149.of_.2026.pdf
--------------------------------------------------
 البدء في معالجة: law_no.286-2021.pdf
 جاري تحويل وتحليل: law_no.286-2021.pdf
--------------------------------------------------
 البدء في معالجة: law.no_.150.of_.2026.pdf
 جاري تحويل وتحليل: law.no_.150.of_.2026.pdf
--------------------------------------------------
 البدء في معالجة: vat-law_no.13-2020.pdf
 جاري تحويل وتحليل: vat-law_no.13-2020.pdf

 تم الانتهاء من جميع الملفات بنجاح!
 إجمالي الفقرات المستخرجة: 311 فقرة جاهزة للـ Vector Store.


****

Synthetic data generation

In [1]:
import os
import random
import arabic_reshaper
from bidi.algorithm import get_display
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

pdfmetrics.registerFont(TTFont('Amiri', 'Amiri-Regular.ttf'))


def ar(text):
    reshaped_text = arabic_reshaper.reshape(str(text))
    return get_display(reshaped_text)

def create_invoice_pdf(filename, invoice_data):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    c.setFont('Amiri', 20)
    c.drawRightString(width - 50, height - 50, ar("فاتورة مبيعات ضريبية"))

    c.setFont('Amiri', 12)
    c.drawRightString(width - 50, height - 80, ar(f"رقم الفاتورة: {invoice_data['invoice_id']}"))
    c.drawRightString(width - 50, height - 100, ar(f"التاريخ: {invoice_data['date']}"))


    c.drawRightString(width - 50, height - 130, ar(f"اسم الشركة الموردة: {invoice_data['supplier_name']}"))
    c.drawRightString(width - 50, height - 150, ar(f"الرقم الضريبي للمورد: {invoice_data['tax_id']}"))


    c.line(50, height - 170, width - 50, height - 170)


    y = height - 200
    c.setFont('Amiri', 14)
    c.drawRightString(width - 50, y, ar("البيان / السلعة"))
    c.drawString(50, y, ar("السعر (ج.م)"))

    c.setFont('Amiri', 12)
    y -= 30
    for item in invoice_data['items']:
        c.drawRightString(width - 50, y, ar(item['name']))
        c.drawString(50, y, f"{item['price']:.2f}")
        y -= 25

    c.line(50, y, width - 50, y)
    y -= 30


    c.setFont('Amiri', 12)
    c.drawRightString(width - 50, y, ar("المبلغ الإجمالي قبل الضريبة:"))
    c.drawString(50, y, f"{invoice_data['subtotal']:.2f}")

    y -= 20
    c.drawRightString(width - 50, y, ar("ضريبة القيمة المضافة (14%):"))
    c.drawString(50, y, f"{invoice_data['vat_amount']:.2f}")

    y -= 25
    c.setFont('Amiri', 14)
    c.drawRightString(width - 50, y, ar("الإجمالي النهائي:"))
    c.drawString(50, y, f"{invoice_data['total']:.2f}")

    c.save()

os.makedirs("synthetic_invoices", exist_ok=True)

companies = ["شركة الأمل للتجارة", "مؤسسة النور للصناعة", "شركة التقنية المصرية", "مجموعة النيل للتوريدات"]
items_pool = [("أجهزة حاسب آلي", 15000), ("طابعات مكتبية", 4500), ("ورق طباعة A4", 800), ("شاشات عرض", 6000)]

for i in range(1, 11):
    inv_id = f"INV-2026-{i:03d}"
    supplier = random.choice(companies)
    selected_item, base_price = random.choice(items_pool)

    if i in [1, 2, 3, 4, 5]:
        tax_id = "123456789"
        subtotal = base_price
        vat = subtotal * 0.14
        total = subtotal + vat
        status = "Valid"

    elif i == 6:
        tax_id = "987654321"
        subtotal = base_price
        vat = subtotal * 0.10
        total = subtotal + vat
        status = "Error_VAT_Math"

    elif i == 7:
        tax_id = "123456"
        subtotal = base_price
        vat = subtotal * 0.14
        total = subtotal + vat
        status = "Error_Invalid_TaxID"

    elif i == 8:
        tax_id = "555666777"
        subtotal = base_price
        vat = subtotal * 0.14
        total = subtotal + vat + 200
        status = "Error_Total_Mismatch"

    else:
        inv_id = "INV-2026-001"
        tax_id = "123456789"
        subtotal = base_price
        vat = subtotal * 0.14
        total = subtotal + vat
        status = "Error_Duplicate"

    data = {
        "invoice_id": inv_id,
        "date": "2026-09-08",
        "supplier_name": supplier,
        "tax_id": tax_id,
        "items": [{"name": selected_item, "price": base_price}],
        "subtotal": subtotal,
        "vat_amount": vat,
        "total": total
    }

    filepath = f"synthetic_invoices/{status}_{inv_id}.pdf"
    create_invoice_pdf(filepath, data)

print(" تم توليد 10 فواتير مصنعة بنجاح داخل مجلد 'synthetic_invoices'!")

 تم توليد 10 فواتير مصنعة بنجاح داخل مجلد 'synthetic_invoices'!


OCR Extractor



In [2]:
import glob
import re
import pdfplumber

processed_invoices_db = []


def extract_invoice_data_definitive(pdf_path):
  with pdfplumber.open(pdf_path) as pdf:

    lines = []
    for page in pdf.pages:
      text = page.extract_text() or ""
      lines.extend([line.strip() for line in text.split("\n") if line.strip()])

  full_text = " ".join(lines)


  inv_id_match = re.search(r"INV-\d{4}-\d{3}", full_text)
  inv_id = inv_id_match.group(0) if inv_id_match else None


  tax_id = None
  tax_match = re.search(r"(\d{6,9})", full_text)

  all_possible_tax = re.findall(r"\b\d{6,9}\b", full_text)
  for t in all_possible_tax:
    if "2026" not in t:
      tax_id = t
      break

  subtotal, vat, total = 0.0, 0.0, 0.0

  for line in lines:

    if "قبل الضريبة" in line or "قبل الضريبه" in line:
      nums = re.findall(r"\d+\.\d+", line)
      if nums:
        subtotal = float(nums[0])

    elif "ضريبة القيمة المضافة" in line or "المضافة" in line:
      nums = re.findall(r"\d+\.\d+", line)
      if nums:

        valid_nums = [n for n in nums if n not in ["14.0", "10.0", "0.14"]]
        if valid_nums:
          vat = float(valid_nums[0])


    elif "الإجمالي النهائي" in line or "الاجمالي النهائي" in line:
      nums = re.findall(r"\d+\.\d+", line)
      if nums:
        total = float(nums[0])


  if subtotal == 0.0 or vat == 0.0 or total == 0.0:
    all_numbers = [
        float(x)
        for x in re.findall(r"\d+\.\d+", full_text)
        if float(x) not in [14.0, 10.0]
    ]

    unique_nums = sorted(list(set(all_numbers)))

    if len(unique_nums) >= 3:

      total = max(unique_nums)

      remaining = [n for n in unique_nums if n != total]
      subtotal = max(remaining)
      vat = min(remaining)

  return {
      "invoice_id": inv_id,
      "tax_id": tax_id,
      "subtotal": subtotal,
      "vat": vat,
      "total": total,
      "file_name": pdf_path,
  }


def run_validation_and_calc_agents(data):
  errors = []

  # 1. Validation Agent = Duplicate Check
  if data["invoice_id"] in processed_invoices_db:
    errors.append(
        " خطأ تكرار: رقم الفاتورة مسجل سابقاً في قاعدة البيانات."
    )
  else:
    processed_invoices_db.append(data["invoice_id"])

  if not data["tax_id"] or len(str(data["tax_id"])) != 9:
    errors.append(
        f" خطأ شكلي: الرقم الضريبي '{data['tax_id']}' غير مكتمل (يجب أن يتكون"
        " من 9 أرقام)."
    )

  # 2. Calculation Agent == Math Validation
  expected_vat = round(data["subtotal"] * 0.14, 2)
  if abs(data["vat"] - expected_vat) > 0.01:
    errors.append(
        f" خطأ حسابي: قيمة الضريبة المحسوبة ({data['vat']}) لا تساوي 14% من"
        f" الصافي {data['subtotal']} (المتوقع: {expected_vat})."
    )

  expected_total = round(data["subtotal"] + data["vat"], 2)
  if abs(data["total"] - expected_total) > 0.01:
    errors.append(
        f" خطأ حسابي: الإجمالي النهائي ({data['total']}) غير مطابق للمجموع"
        f" الحقيقي (المتوقع: {expected_total})."
    )

  return errors



processed_invoices_db = []
all_pdf_invoices = sorted(glob.glob("synthetic_invoices/*.pdf"))

for pdf in all_pdf_invoices:
  inv_data = extract_invoice_data_definitive(pdf)
  detected_errors = run_validation_and_calc_agents(inv_data)

  print(f"\n جاري فحص: {inv_data['file_name']}")
  if not detected_errors:
    print(" الفاتورة سليمة ومطابقة لكافة معايير الامتثال!")
  else:
    for err in detected_errors:
      print(f"  {err}")


 جاري فحص: synthetic_invoices/Error_Duplicate_INV-2026-001.pdf
 الفاتورة سليمة ومطابقة لكافة معايير الامتثال!

 جاري فحص: synthetic_invoices/Error_Invalid_TaxID_INV-2026-007.pdf
   خطأ شكلي: الرقم الضريبي '123456' غير مكتمل (يجب أن يتكون من 9 أرقام).

 جاري فحص: synthetic_invoices/Error_Total_Mismatch_INV-2026-008.pdf
   خطأ حسابي: الإجمالي النهائي (7040.0) غير مطابق للمجموع الحقيقي (المتوقع: 6840.0).

 جاري فحص: synthetic_invoices/Error_VAT_Math_INV-2026-006.pdf
   خطأ حسابي: قيمة الضريبة المحسوبة (80.0) لا تساوي 14% من الصافي 800.0 (المتوقع: 112.0).

 جاري فحص: synthetic_invoices/Valid_INV-2026-001.pdf
   خطأ تكرار: رقم الفاتورة مسجل سابقاً في قاعدة البيانات.

 جاري فحص: synthetic_invoices/Valid_INV-2026-002.pdf
 الفاتورة سليمة ومطابقة لكافة معايير الامتثال!

 جاري فحص: synthetic_invoices/Valid_INV-2026-003.pdf
 الفاتورة سليمة ومطابقة لكافة معايير الامتثال!

 جاري فحص: synthetic_invoices/Valid_INV-2026-004.pdf
 الفاتورة سليمة ومطابقة لكافة معايير الامتثال!

 جاري فحص: synthetic_invo

In [3]:
def run_audit_agents(data, processed_db):
  errors = []

  # Validation Agent
  if data["invoice_id"] in processed_db:
    errors.append("Duplicate invoice ID detected in database")
  else:
    processed_db.append(data["invoice_id"])

  if not data["tax_id"] or len(str(data["tax_id"])) != 9:
    errors.append(f"Invalid Tax ID format: {data['tax_id']}")

  # Calculation Agent
  expected_vat = round(data["subtotal"] * 0.14, 2)
  if abs(data["vat"] - expected_vat) > 0.01:
    errors.append(
        f"VAT miscalculation: found {data['vat']} expected {expected_vat}"
    )

  expected_total = round(data["subtotal"] + data["vat"], 2)
  if abs(data["total"] - expected_total) > 0.01:
    errors.append(
        f"Total mismatch: found {data['total']} expected {expected_total}"
    )

  return errors

Retrieval-Augmented Generation

In [4]:
def get_legal_explanation(error_msg):
  results = collection.query(query_texts=[error_msg], n_results=1)

  if not results["documents"][0]:
    return "No legal match found"

  context = results["documents"][0][0]
  metadata = results["metadatas"][0][0]

  augmented_prompt = f"""
    Context: {context}
    Law: {metadata.get('law_name')}
    Article: {metadata.get('article_number')}
    Violation: {error_msg}
    Provide a concise legal opinion.
    """
#generate_content
  response = client_gemini.models.generate_content(
      model="gemini-3.6-flash", contents=augmented_prompt
  )
  return response.text

Set API Keys and Load Documents

In [5]:
import os
import warnings
from google.colab import userdata

warnings.filterwarnings("ignore")

GOOGLE_KEY = userdata.get("GEMINI_API_KEY")
TAVILY_KEY = userdata.get("TAVILY_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_KEY


#Document Processing
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

loader = PyPDFDirectoryLoader(".")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs)

# --- Embeddings و Vectorstore ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="legal-invoice-chroma",
    embedding=embeddings,
)

retriever = vectorstore.as_retriever()

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GOOGLE_KEY,
    temperature=0
)


print("AII TOOLS IS DONE")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AII TOOLS IS DONE


Create Grader, RAG Chain, and Web Search Tools (Corrective RAG - CRAG).

قراءة جميع الصفحات والنصوص وتحويلها إلى مستندات برمجية.منساش

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel, Field

class GradeDocuments(BaseModel):
    binary_score: str = Field(description="Relevance check on retrieved legal or invoice document, 'yes' or 'no'")

structured_llm_grader = llm.with_structured_output(GradeDocuments)

system_grader = """You are a grader assessing whether a retrieved legal or invoice document is relevant to the user query or violation.
If the document contains matching laws, article numbers, or relevant invoice details, grade it as 'yes', otherwise 'no'"""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_grader),
    ("human", "Retrieved document: \n\n {document} \n\n User query: {question}"),
])
retrieval_grader = grade_prompt | structured_llm_grader

system_rag = """You are a legal and financial assistant. Provide a concise legal opinion using the provided context, mentioning the law and article if available.
If context is insufficient, state that clearly."""

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_rag),
    ("human", "Context: {context} \n\n User query: {question} \n\n Provide a concise legal opinion:")
])
rag_chain = prompt_rag | llm | StrOutputParser()

system_rewriter = """You are an AI assistant that rewrites error messages or legal violation queries into clear queries for web search."""

re_write_prompt = ChatPromptTemplate.from_messages([
    ("system", system_rewriter),
    ("human", "Initial query: \n\n {question} \n Formulate an improved web search query."),
])
question_rewriter = re_write_prompt | llm | StrOutputParser()

web_search_tool = TavilySearchResults(k=3)
# Re-initializing components after LLM update

/tmp/ipykernel_9262/1384660910.py:37: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(k=3)


Define Graph State and Nodes

In [7]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document

class GraphState(TypedDict):
    question: str
    generation: str
    web_search: str
    documents: List[Document]

def retrieve(state: GraphState):
    print("---RETRIEVE DOCUMENTS---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

def grade_documents(state: GraphState):
    print("---CHECK RELEVANCE OF DOCUMENTS---")
    question = state["question"]
    documents = state["documents"]

    filtered_docs = []
    web_search_flag = "No"

    for d in documents:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        if score.binary_score.lower() == "yes":
            print("---GRADE: MATCHING DOCUMENT FOUND---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search_flag = "Yes"

    if not filtered_docs:
        web_search_flag = "Yes"

    return {"documents": filtered_docs, "question": question, "web_search": web_search_flag}

def transform_query(state: GraphState):
    print("---TRANSFORM QUERY FOR EXTERNAL SEARCH---")
    question = state["question"]
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": state["documents"], "question": better_question}

def web_search(state: GraphState):
    print("---WEB SEARCH FOR MISSING DATA---")
    question = state["question"]
    documents = state.get("documents", [])

    docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in docs])
    web_results_doc = Document(page_content=web_results)

    documents.append(web_results_doc)
    return {"documents": documents, "question": question}

def generate(state: GraphState):
    print("---GENERATE RESPONSE---")
    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join([d.page_content for d in documents])
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"documents": documents, "question": question, "generation": generation}

def decide_to_generate(state: GraphState):
    if state["web_search"] == "Yes":
        print("---DECISION: LOCAL DATA INSUFFICIENT, SWITCH TO WEB SEARCH---")
        return "transform_query"
    else:
        print("---DECISION: SUFFICIENT DATA, GENERATE RESPONSE---")
        return "generate"

Build LangGraph Workflow

In [8]:
from langgraph.graph import END, StateGraph

workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search_node", web_search)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
    },
)
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
# Re-compiling workflow after component updates

Wrapper Function and Execution

In [9]:
def get_legal_explanation(error_msg: str):
    inputs = {"question": error_msg}
    final_output = None
    for output in app.stream(inputs):
        for key, value in output.items():
            final_output = value
    if final_output and "generation" in final_output:
        return final_output["generation"]
    return "No legal match found."

# Test call
error_message = "Invoice missing tax registration number"
result = get_legal_explanation(error_message)
print("\n--- RESULT ---")
print(result)
# Re-running test call after workflow re-compilation

---RETRIEVE DOCUMENTS---
---CHECK RELEVANCE OF DOCUMENTS---
---GRADE: MATCHING DOCUMENT FOUND---
---GRADE: MATCHING DOCUMENT FOUND---
---GRADE: MATCHING DOCUMENT FOUND---
---GRADE: MATCHING DOCUMENT FOUND---
---DECISION: SUFFICIENT DATA, GENERATE RESPONSE---
---GENERATE RESPONSE---

--- RESULT ---
Based on the provided context, the information is **insufficient** to deliver a formal legal opinion or cite specific laws and articles. 

The context contains only transaction items, pre-tax amounts, VAT amounts, and total totals, but does not include legal provisions, tax code references, or regulations regarding mandatory invoice requirements (such as tax registration numbers).


#**Function** to be used in Streamlit app.py

In [10]:
def get_legal_explanation(error_msg: str):
    inputs = {"question": error_msg}
    final_output = None
    for output in app.stream(inputs):
        for key, value in output.items():
            final_output = value
    if final_output and "generation" in final_output:
        return final_output["generation"]
    return "No legal match found."

In [11]:
!pip freeze > requirements.txt